# MetaMARL visualization module tutorial

This tutorial explains how the fishery experiment turns the numbers produced by its environments and optimizers into plots and files. The visualization stack separates four concerns:

```
WHAT DATA EXISTS        -> MetricSchema
HOW DATA ACCUMULATES    -> Metric / MetricLogger
WHAT DATA TO SELECT     -> Query
HOW IT IS DISPLAYED     -> Reporter
```

The environment, the inner optimizer and the outer optimizer therefore contain no plotting code. They fill typed metrics. A query selects paths inside those metrics, and a reporter renders the selection to a backend: a CSV file per query, a TensorBoard event directory or a Weights & Biases run.

The notebook covers, in order:

1. how a `Query` is written, and how the `ReduceProtocol` tokens `SERIES` and `MEAN` expand the dynamic parts of a path (mechanism, seed, episode, policy, agent, parameter);
2. how the metric logger accumulates values, and the difference between `peek()` and `reduce()`;
3. where the reporting of an experiment is configured, and the schema tree that the paths refer to;
4. how to add a metric at the right level of the tree;
5. query sets for the environment, the inner (RLlib) optimizer and the outer (ES) optimizer, each checked against populated schemas;
6. the query sets shipped with the fishery example, and what a stale path looks like when it fails;
7. the CSV backend, the reporting lifecycle, and the limits of the present design.

The notebook runs in a few seconds and starts no Ray cluster. The metrics it plots are made-up numbers, produced by a small simulation that pushes values into the real schemas and loggers in the same way as the real environment and optimizers do. The shapes are the real ones, but the values carry no scientific meaning. For a real training run, see `tutorials/metamarl_fishery_tutorial.ipynb` and `examples/bilevel_fishery/debug.py`.

The `examples/` directory is not an installed package, so the first code cell puts the repository root on the import path. The cell also makes the notebook independent of the directory it is started from.

In [ ]:
import sys
from pathlib import Path

# `examples/` is not an installed package: put the repository root on the path.
REPO_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

## Imports and helpers

The imports below cover the schemas, the logger, the query and reporter classes, and the fishery example. Logging is switched on at the INFO level because a reporter never lets a failing query abort a run: it logs the traceback and moves on, so the log is where a stale query becomes visible.

In [ ]:
import logging
import tempfile
import typing
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from pydantic import Field

from core.adaptors.ray.schema import (
    EvalSchema,
    LearnerSchema,
    MechanismLearnerSchema,
    MechanismRolloutSchema,
    PerformanceSchema,
    PolicyLearnerSchema,
    RaySchema,
    RolloutSchema,
    SeedLearnerSchema,
    SeedRolloutSchema,
    TrainSchema,
)
from core.envs.schema import EpisodeRolloutSchema
from core.metrics.enums import ReduceProtocol
from core.metrics.logger import MetricLogger
from core.metrics.schemas import MetricSchema
from core.optimizers.es.schema import ESCandidateSchema, ESParameterSchema, ESSchema
from core.reporting.base import Reporter
from core.reporting.csv import CSVConfig
from core.reporting.query import Path as MetricPath
from core.reporting.query import Query
from examples.bilevel_fishery.metric_schema import FisheryMetricSchema
from examples.bilevel_fishery.queries import (
    ES_QUERIES,
    FISHERY_ENV_QUERIES,
    INNER_QUERIES,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

# Short names for the two reduction tokens used inside query paths.
SERIES = ReduceProtocol.SERIES
MEAN = ReduceProtocol.MEAN

## 1. A query is a set of paths

A `Query` points at metric series through the schema tree. A path is a tuple of field names, so a one-element path needs a trailing comma: `("iter",)` is a path, whereas `("iter")` is just the string `"iter"`.

```python
Query(
    title: str,
    x: Path,
    y: Path | tuple[Path, ...],
    x_label=None, y_label=None,
    legend_labels=None, plot_modes=None, show_group_labels=True,
    color=None, color_label=None, colorscale=None,
    error="none", error_path=None,
)
```

Several `y` paths are drawn in one plot, and `legend_labels` and `plot_modes` carry one entry per path. The `color` path colours the points of every series by a third series, which is how the scatter of candidate fitness against a parameter is coloured by generation.

A path walks the schema one token at a time. At a *dynamic* node, which is a dictionary keyed by a runtime identifier (`by_mechanism`, `by_seed`, `by_episode`, `by_policy`, `by_agent`, `by_parameter`), the next token says what to do with the keys:

- `ReduceProtocol.SERIES` keeps one series per key, with the keys sorted as text (so `"10"` comes before `"2"`), so a plot gets one curve per mechanism, per seed or per agent;
- `ReduceProtocol.MEAN` averages the keys point by point and skips the gaps that the logger records for a key that was absent from a push.

A concrete key such as `"0"` is also accepted in place of a token, but it ties the query to one run, so the tokens are preferred. There is no wildcard character.

The reduction is therefore a token inside the path, not an argument of the query. A mean with a standard-deviation band takes two pieces: a `MEAN` token in a `y` path, and `error="std"` with an `error_path` that names the dynamic node whose branches are averaged. The band is the population standard deviation across those branches, and `error_path` must stop before the `MEAN` token it refers to.

In [ ]:
# One series: the path ends on a field of the root schema.
single = Query(title="Fish stock", x=("iter",), y=("fish_stock",))

# One series per mechanism: SERIES expands the dynamic node `by_mechanism`.
per_mechanism = Query(
    title="Fitness per candidate", x=("iter",), y=("by_mechanism", SERIES, "fitness")
)

# The mean over mechanisms, with one standard deviation across them as a band.
mean_band = Query(
    title="Mean fitness",
    x=("iter",),
    y=("by_mechanism", MEAN, "fitness"),
    error="std",
    error_path=("by_mechanism",),
)

for query in (single, per_mechanism, mean_band):
    print(f"{query.title:<24} y paths: {query.y_paths}")

# A band that has no MEAN token to take it over is rejected when the query is
# declared, not at the end of an iteration.
try:
    Query(
        title="Bad band",
        x=("iter",),
        y=("by_mechanism", SERIES, "fitness"),
        error="std",
        error_path=("by_mechanism",),
    )
except ValueError as error:
    print("\nValueError:", error)

## 2. Accumulating metrics: `peek()` versus `reduce()`

A `MetricLogger` mirrors a schema and owns the accumulated values. It is fed with `push(path, value)` for one leaf or `push_data(schema_instance)` for a whole payload, and it is read back in one of two ways:

- `peek()` returns the raw history of every leaf as a list, whatever the leaf's reduction, and keeps it. Reporters resolve their queries against a peeked schema, so every leaf of it is a list with one entry per push.
- `reduce()` collapses each leaf according to its declared reduction (a mean, a maximum, the last value, the whole series), returns the result as a schema instance of scalars, and clears the accumulators.

A schema can also override the reduction of a whole subtree. `RaySchema.train` and `RaySchema.eval` declare the subtree reduction `LAST`, which replaces the declared reduction of every leaf under them, so reducing the history of an inner run keeps the values of its last iteration and nothing else.

The cell below shows both on a toy schema.

In [ ]:
class Toy(MetricSchema):
    """Two leaves with different reductions, to contrast peek and reduce."""

    loss: Optional[float] = Field(
        default=None, json_schema_extra={"reduce": ReduceProtocol.MEAN}
    )
    best: Optional[float] = Field(
        default=None, json_schema_extra={"reduce": ReduceProtocol.MAX}
    )


toy_logger = MetricLogger.from_schema(Toy)

for loss in (3.0, 2.0, 1.0):
    toy_logger.push_data(Toy(loss=loss, best=-loss))

peeked = toy_logger.peek()
print("peek   -> loss:", peeked.loss, "| best:", peeked.best)

reduced = toy_logger.reduce()
print("reduce -> loss:", reduced.loss, "| best:", reduced.best)

print("peek after reduce -> loss:", toy_logger.peek().loss)

assert peeked.loss == [3.0, 2.0, 1.0]
assert reduced.loss == 2.0 and reduced.best == -1.0
assert toy_logger.peek().loss == []

The distinction decides when each level reports. The ES optimizer keeps its logger alive for the whole search and calls `peek()` after every generation, so its plots are cumulative over generations and the reporter needs no history cache of its own. The environment reports the history of an episode with `peek()` and only then reduces it to the per-episode summary that is handed upward. Section 12 follows the three levels through a run.

## 3. Where reporting is configured

A bilevel run has three reporting levels, and one backend serves all of them. (The bilevel configuration also builds a primary reporter of its own for the run, which receives no queries in this example.) The backend is chosen once on the bilevel configuration, which copies the configuration into every level so that each environment and optimizer builds its own reporter. A level only declares its own queries.

```python
BilevelConfig()
.reporter(config=CSVConfig(project="bilevel_fishery"))   # or WandbConfig / TensorBoardConfig
.regulator(
    ESConfig()
    ...
    .reporting(queries=ES_QUERIES)                        # outer level: paths on ESSchema
)
.society(
    APPOptimizerConfig()
    ...
    .environment(
        env=FisheryRegulatedEnv,
        ...,
        schema=FisheryMetricSchema,                       # environment level
        queries=FISHERY_ENV_QUERIES,                      # paths on FisheryMetricSchema
    )
    .reporting(queries=INNER_QUERIES)                     # inner level: paths on RaySchema
)
```

The fishery example gives a schema only to the environment, because it is the only level whose metrics the example defines; the base optimizer configuration lets any environment declare one. The two optimizers log into a schema fixed by their class, `ESSchema` for the evolution strategy and `RaySchema` for the RLlib society, so `.reporting(queries=...)` needs no schema: the paths of its queries simply refer to that one. The three levels differ as follows.

- **Environment level.** The queries run on one `FisheryMetricSchema` history, a time series over the steps of one episode. They are the horizon plots: stock, growth, harvest and the per-fisher requests.
- **Inner optimizer level.** The queries run on a `RaySchema` that holds the training iterations of one inner training run: rollout aggregates, per-mechanism, per-seed and per-episode values, learner statistics and performance counters.
- **Outer optimizer level.** The queries run on an `ESSchema` that holds one entry per generation: the fitness of every candidate, the search distribution, and, under `inner`, the reduced `RaySchema` of the inner run of each generation.

`debug.py` wires `ES_QUERIES` and `INNER_QUERIES` and leaves the environment queries empty. `FISHERY_ENV_QUERIES` holds one environment query that is available to wire in as shown above.

## 4. The schema tree and where to add a metric

The level of a field in the schema tree decides its path and its meaning. The helper below prints every dynamic node (a dictionary keyed by a runtime identifier) of the three schemas of this experiment, which is where the `SERIES` and `MEAN` tokens apply. A path may continue through a nested schema with plain field names and must cross a dynamic node with a token or a key.

In [ ]:
def unwrap(annotation):
    """Strip Optional from an annotation."""
    if typing.get_origin(annotation) in (typing.Union, type(int | None)):
        args = [a for a in typing.get_args(annotation) if a is not type(None)]
        return unwrap(args[0])
    return annotation


def dynamic_nodes(schema: type[MetricSchema], prefix: str = "") -> list[str]:
    """List the dotted paths of the dictionary nodes under a schema class."""
    lines: list[str] = []
    for name, field in schema.model_fields.items():
        annotation = unwrap(field.annotation)
        if typing.get_origin(annotation) is dict:
            value_type = unwrap(typing.get_args(annotation)[1])
            lines.append(f"{prefix}{name}[*]  ->  {value_type.__name__}")
            lines += dynamic_nodes(value_type, f"{prefix}{name}[*].")
        elif isinstance(annotation, type) and issubclass(annotation, MetricSchema):
            lines += dynamic_nodes(annotation, f"{prefix}{name}.")
    return lines


for schema in (FisheryMetricSchema, RaySchema, ESSchema):
    print(schema.__name__)
    for line in dict.fromkeys(dynamic_nodes(schema)):
        print("   ", line)

Three details of this tree matter when a query is written.

- `RaySchema.train.rollout` has an `aggregate` (one `EpisodeRolloutSchema` for the whole iteration, over all mechanisms) next to `by_mechanism`. Below `by_mechanism`, a seed holds only `by_episode`: there is no `aggregate` under `by_seed`, so the mean over the episodes of a seed is taken by a `MEAN` token on `by_episode`.
- Each `by_episode` value is the reduced `FisheryMetricSchema` of one episode. Its `*_series` fields keep every step of the episode, so inside the logger of the inner optimizer a peeked `*_series` leaf is a list of lists, one inner list per iteration. A path that ends on such a leaf is rejected as a nested series; plot the scalar summaries (`fish_norm_next_mean`, `fish_norm_next_last`, and so on) instead.
- `ESSchema.inner` is declared as the generic `MetricSchema`. The logger specializes it at runtime to the schema of the object that is pushed (`RaySchema` for an RLlib inner optimizer), so the ES schema does not need to know which optimizer it wraps. Likewise `EpisodeRolloutSchema` is specialized to `FisheryMetricSchema` below `by_episode`. The runtime schema must be a subclass of the declared one.

The reductions that a field can declare are the members of `ReduceProtocol`.

In [ ]:
names = [member.name for member in ReduceProtocol]
print(names)
assert set(names) == {"MEAN", "SUM", "MAX", "MIN", "LAST", "EMA", "SERIES"}

`EMA` is declared but has no implementation yet, and building a logger for a schema that uses it raises `NotImplementedError`. `SERIES` keeps the history itself as the value, so the reduced leaf is the list of pushed values. The others compile a history into one number.

### Adding a metric

To add a metric, first decide which entity owns it, then add a field to the schema of that entity with the reduction that suits it. The path of the new leaf follows from where it sits in the tree.

- A quantity of the shared environment, such as a stock or a harvest total, goes on the environment's episode schema (`FisheryMetricSchema`). Its path is the field name, and below `by_episode` of the inner optimizer it becomes `(..., "by_episode", MEAN, "<field>")`.
- A quantity that differs by agent goes on the per-agent schema (`FisheryAgentMetricSchema`, the value type of `by_agent`). Its path is `("by_agent", SERIES, "<field>")` for one curve per fisher, or `("by_agent", MEAN, "<field>")` for their mean.
- A learner statistic goes on `PolicyLearnerSchema`, reached through `("train", "learner", "by_mechanism", SERIES, "by_seed", MEAN, "by_policy", SERIES, "<field>")`.
- A throughput or timing counter goes on `PerformanceSchema`, reached through `("train", "performance", "<field>")`.
- A quantity that should be plotted across outer generations goes on `ESSchema` with the `SERIES` reduction, so that every push appends one value and the logger, which lives for the whole search, grows the series.

Declaring the field is enough for the logger and the queries. The producer of the metric (the environment, the Ray adaptor or the ES optimizer) must also push a value for it, otherwise a query on the field resolves to an empty series. The cell below adds a made-up catch-per-unit-effort field to a subclass of the fishery schema and pushes it, and Section 6 then checks that a query on the new leaf resolves.

In [ ]:
class CatchPerEffortSchema(FisheryMetricSchema):
    """The fishery schema plus one made-up per-step metric."""

    catch_per_effort: Optional[float] = Field(
        default=None, json_schema_extra={"reduce": ReduceProtocol.MEAN}
    )


extended_logger = MetricLogger.from_schema(CatchPerEffortSchema)
for step, value in enumerate((12.0, 11.5, 11.9), start=1):
    extended_logger.push(("iter",), step)
    extended_logger.push(("catch_per_effort",), value)

extended_history = extended_logger.peek()
extended_query = Query(
    title="Catch per unit effort", x=("iter",), y=("catch_per_effort",)
)

print("new leaf in the history:", extended_history.catch_per_effort)
assert extended_history.catch_per_effort == [12.0, 11.5, 11.9]

## 5. Made-up metrics for the examples

To exercise queries without starting Ray, the cells below simulate a small fishery and push its values into the real loggers. The simulation is a simplified version of `FisheryRegulatedEnv`. It uses a logistic stock (Schaefer, 1954), `B(t+1) = max(B + g(B) + noise - C, 0)`, without the restoration term of the real environment, and rations simultaneous requests pro rata. It fills only part of the fields of the fishery schema: the stock-level fields, the reference points and the per-fisher requested and delivered harvest, with an invented reward (the realized harvest over the maximum sustainable yield). It does not fill the identifier fields. The reference points $B_{\mathrm{MSY}} = K/2$ and $\mathrm{MSY} = rK/4$ are those of the logistic model (Schaefer, 1954; Pella and Tomlinson, 1969, for the general shape parameter). An episode is first logged step by step (what the environment's reporter receives), then reduced (what the inner optimizer receives).

The values are invented and the constants below are reduced on purpose, so that the whole notebook runs in seconds: four candidates, two training seeds, three fishers, 30 steps per episode, six inner iterations and six ES generations. The "candidates" differ only by the harvest effort that their quota allows.

In [ ]:
SEED = 7
N_MECHANISMS = 4  # candidates per ES generation
SEEDS = ("11", "22")  # training seeds of each candidate
EVAL_SEEDS = ("901", "902")  # evaluation seeds of each candidate
N_FISHERS = 3
HORIZON = 30  # steps per episode
N_TRAIN_ITERATIONS = 6  # inner training iterations per ES generation
N_GENERATIONS = 6

CARRYING_CAPACITY = 5_000.0  # K, biomass units
GROWTH_RATE = 0.3  # r, per step
PROCESS_NOISE = 0.05  # sigma, relative to the stock
# Reference points of a logistic stock (the shape parameter p equals one).
B_MSY = CARRYING_CAPACITY / 2.0
MSY = GROWTH_RATE * CARRYING_CAPACITY / 4.0
F_MSY = MSY / B_MSY

In [ ]:
def log_episode(*, harvest_level: float, seed: int) -> MetricLogger:
    """Log one made-up episode step by step, as the environment does.

    Parameters
    ----------
    harvest_level : float
        Harvest effort of the fishers as a multiple of the rate at MSY.
    seed : int
        Seed of the noise of the episode.

    Returns
    -------
    MetricLogger
        A ``FisheryMetricSchema`` logger holding ``HORIZON`` pushes per leaf.
    """
    rng = np.random.default_rng(seed)
    logger = MetricLogger.from_schema(FisheryMetricSchema)
    biomass = 0.7 * CARRYING_CAPACITY

    for step in range(HORIZON):
        shares = np.clip(1.0 + 0.1 * rng.standard_normal(N_FISHERS), 0.0, None)
        requests = harvest_level * F_MSY * biomass / N_FISHERS * shares
        attempted = float(requests.sum())
        realized = min(attempted, biomass)
        delivered = requests * (realized / attempted if attempted > 0.0 else 0.0)
        noise = float(PROCESS_NOISE * rng.standard_normal() * biomass)
        growth = GROWTH_RATE * biomass * (1.0 - biomass / CARRYING_CAPACITY) + noise
        next_biomass = max(biomass + growth - realized, 0.0)
        normalized = next_biomass / CARRYING_CAPACITY
        reward = realized / MSY

        logger.push(("iter",), step + 1)
        values = {
            "B_msy": B_MSY,
            "MSY": MSY,
            "F_msy": F_MSY,
            "fish_stock": biomass,
            "fish_stock_next": next_biomass,
            "growth": growth,
            "growth_noise": noise,
            "H_attempted": attempted,
            "H_realized": realized,
            "total_usage_norm": realized / CARRYING_CAPACITY,
            "fish_norm_next_mean": normalized,
            "fish_norm_next_min": normalized,
            "fish_norm_next_max": normalized,
            "fish_norm_next_last": normalized,
            "fish_norm_next_series": normalized,
            "H_realized_series": realized,
            "reward_series": reward,
            "reward_total": reward,
            "reward_mean": reward,
            "reward_min": reward,
            "reward_max": reward,
        }
        for name, value in values.items():
            logger.push((name,), value)
        for index in range(N_FISHERS):
            agent = f"fisherman:{index}"
            logger.push(
                ("by_agent", agent, "requested_harvest"), float(requests[index])
            )
            logger.push(
                ("by_agent", agent, "delivered_harvest"), float(delivered[index])
            )
        biomass = next_biomass

    return logger

In [ ]:
def rollout_payload(
    levels: dict[str, float], seeds: tuple[str, ...], *, run_seed: int, lifetime: int
) -> RolloutSchema:
    """Build the rollout branch of one RLlib iteration from made-up episodes.

    Parameters
    ----------
    levels : dict[str, float]
        Harvest effort of each candidate, keyed by its mechanism id.
    seeds : tuple[str, ...]
        Seed ids simulated for every candidate.
    run_seed : int
        Seed that makes the iteration reproducible.
    lifetime : int
        Number of episodes finished before this iteration.

    Returns
    -------
    RolloutSchema
        An ``aggregate`` over all episodes and the ``by_mechanism`` tree down
        to the reduced ``FisheryMetricSchema`` of every episode.
    """
    by_mechanism: dict[str, MechanismRolloutSchema] = {}
    episodes: list[FisheryMetricSchema] = []

    for mechanism, level in levels.items():
        by_seed = {}
        for seed in seeds:
            episode_seed = run_seed * 1_000 + 100 * int(mechanism) + int(seed)
            episode = log_episode(harvest_level=level, seed=episode_seed).reduce()
            episodes.append(episode)
            episode_id = f"env=0|m={mechanism}|ps={run_seed}|ss={seed}|n=0"
            by_seed[seed] = SeedRolloutSchema(by_episode={episode_id: episode})
        by_mechanism[mechanism] = MechanismRolloutSchema(by_seed=by_seed)

    aggregate = EpisodeRolloutSchema(
        reward_mean=float(np.mean([e.reward_mean for e in episodes])),
        reward_min=float(np.min([e.reward_min for e in episodes])),
        reward_max=float(np.max([e.reward_max for e in episodes])),
        episode_len_mean=float(HORIZON),
        episode_len_min=float(HORIZON),
        episode_len_max=float(HORIZON),
        num_episodes=len(episodes),
        num_episodes_lifetime=lifetime + len(episodes),
    )

    return RolloutSchema(aggregate=aggregate, by_mechanism=by_mechanism)


def train_society(levels: dict[str, float], *, run_seed: int) -> MetricLogger:
    """Mimic ``RayOptimizer.train``: N training pushes, then one evaluation push.

    The real optimizer pushes the iteration index and one ``train`` payload per
    iteration, then one ``eval`` payload after the loop. The learner and
    performance values are made up.

    Parameters
    ----------
    levels : dict[str, float]
        Harvest effort of each candidate, keyed by its mechanism id.
    run_seed : int
        Seed that makes the run reproducible.

    Returns
    -------
    MetricLogger
        A ``RaySchema`` logger holding the history of the run (not reduced).
    """
    rng = np.random.default_rng(run_seed)
    logger = MetricLogger.from_schema(RaySchema)
    episodes_done = 0
    steps_done = 0

    for iteration in range(N_TRAIN_ITERATIONS):
        rollout = rollout_payload(
            levels, SEEDS, run_seed=run_seed * 100 + iteration, lifetime=episodes_done
        )
        n_episodes = rollout.aggregate.num_episodes
        episodes_done += n_episodes
        steps_this_iter = n_episodes * HORIZON
        steps_done += steps_this_iter
        decay = np.exp(-0.3 * iteration)

        learner = LearnerSchema(
            by_mechanism={
                mechanism: MechanismLearnerSchema(
                    by_seed={
                        seed: SeedLearnerSchema(
                            by_policy={
                                "fisher_policy": PolicyLearnerSchema(
                                    value_loss=float(decay * (1 + rng.random())),
                                    total_loss=float(decay * (2 + rng.random())),
                                    policy_loss=float(0.1 * rng.standard_normal()),
                                    policy_entropy=float(1.0 - 0.05 * iteration),
                                )
                            }
                        )
                        for seed in SEEDS
                    }
                )
                for mechanism in levels
            }
        )
        performance = PerformanceSchema(
            env_steps_this_iter=float(steps_this_iter),
            env_steps_lifetime=float(steps_done),
            agent_steps_this_iter_sum=float(steps_this_iter * N_FISHERS),
            agent_steps_lifetime_sum=float(steps_done * N_FISHERS),
            env_steps_throughput=float(800 + 100 * rng.random()),
            training_iteration_s=float(0.3 + 0.05 * rng.random()),
            sample_s=float(0.28 + 0.05 * rng.random()),
            learner_update_s=float(0.02 + 0.005 * rng.random()),
        )
        logger.push(("iter",), iteration)
        logger.push_data(
            RaySchema(
                train=TrainSchema(
                    rollout=rollout, learner=learner, performance=performance
                )
            )
        )

    evaluation = rollout_payload(
        levels, EVAL_SEEDS, run_seed=run_seed * 100 + 99, lifetime=0
    )
    logger.push_data(
        RaySchema(
            eval=EvalSchema(
                rollout=evaluation,
                performance=PerformanceSchema(
                    env_steps_this_iter=float(len(levels) * len(EVAL_SEEDS) * HORIZON)
                ),
            )
        )
    )

    return logger

In [ ]:
def run_search() -> tuple[MetricLogger, RaySchema]:
    """Mimic ``ESOptimizer.train`` with a toy search over one parameter.

    Each generation samples ``N_MECHANISMS`` candidates of a normalized
    ``quota`` parameter around the search mean, evaluates them with
    :func:`train_society`, and pushes one ``ESSchema`` payload of the same
    shape as the real optimizer's, with the reduced inner metrics under
    ``inner``. The fitness and the update of the mean are toys.

    Returns
    -------
    tuple[MetricLogger, RaySchema]
        The ES logger after the last generation, and the peeked history of the
        inner run of the first generation, which is what ``RayOptimizer``
        reports at the end of its ``train``.
    """
    rng = np.random.default_rng(SEED)
    logger = MetricLogger.from_schema(ESSchema)
    society_history = None
    mean, best_fitness, best_candidate = 0.5, -np.inf, 0.5

    for generation in range(N_GENERATIONS):
        quota = np.clip(mean + 0.2 * rng.standard_normal(N_MECHANISMS), 0.0, 1.0)
        levels = {str(i): 0.5 + 1.5 * float(q) for i, q in enumerate(quota)}
        society = train_society(levels, run_seed=generation)
        if society_history is None:
            society_history = society.peek()
        inner = society.reduce()

        fitness = np.zeros(N_MECHANISMS)
        for index in range(N_MECHANISMS):
            seeds = inner.train.rollout.by_mechanism[str(index)].by_seed
            episodes = [e for s in seeds.values() for e in s.by_episode.values()]
            fitness[index] = np.mean(
                [e.reward_mean * e.fish_norm_next_last for e in episodes]
            )

        best = int(np.argmax(fitness))
        if fitness[best] > best_fitness:
            best_fitness, best_candidate = float(fitness[best]), float(quota[best])

        logger.push_data(
            ESSchema(
                iter=generation,
                generation=generation,
                sigma=0.2,
                population_size=N_MECHANISMS,
                fitness_mean=float(fitness.mean()),
                fitness_best=float(fitness[best]),
                best_mechanism_idx=best,
                best_fitness_global=best_fitness,
                by_mechanism={
                    str(i): ESCandidateSchema(
                        fitness=float(fitness[i]),
                        by_parameter={
                            "quota": ESParameterSchema(value=float(quota[i]))
                        },
                    )
                    for i in range(N_MECHANISMS)
                },
                search_mean={"quota": ESParameterSchema(value=float(mean))},
                global_best={"quota": ESParameterSchema(value=best_candidate)},
                generation_best={"quota": ESParameterSchema(value=float(quota[best]))},
                inner=inner,
            )
        )
        top = np.argsort(fitness)[-2:]
        mean = float(0.5 * mean + 0.5 * quota[top].mean())

    return logger, society_history


es_logger, ray_history = run_search()
es_history = es_logger.peek()
env_history = log_episode(harvest_level=1.2, seed=3).peek()

print(
    "environment history:",
    type(env_history).__name__,
    "- points per leaf:",
    len(env_history.fish_stock),
)
print(
    "inner (Ray) history:",
    type(ray_history).__name__,
    "- training iterations:",
    len(ray_history.iter),
    "- evaluation points:",
    len(ray_history.eval.rollout.aggregate.reward_mean),
)
print(
    "outer (ES) history :",
    type(es_history).__name__,
    "- generations:",
    len(es_history.generation),
)
print("ESSchema.inner was specialized at runtime to:", type(es_history.inner).__name__)

assert len(env_history.fish_stock) == HORIZON
assert len(ray_history.iter) == N_TRAIN_ITERATIONS
assert len(ray_history.eval.rollout.aggregate.reward_mean) == 1
assert len(es_history.generation) == N_GENERATIONS
assert isinstance(es_history.inner, RaySchema)

The last line shows the runtime specialization at work: `ESSchema.inner` is declared as `MetricSchema`, and the logger replaced it with `RaySchema` when the first payload arrived. The three histories are the objects a reporter receives at each level.

## 6. Drawing and checking queries

A reporter backend has two jobs, and `Reporter` does the first: it walks every path against the history, checks that the x, y, error and colour series of a group have the same length, and passes the resolved series to a backend hook. A subclass only implements that hook (`_report`) and `close`. The class below draws every query into a matplotlib figure. It is a notebook-sized backend: it honours `legend_labels`, `plot_modes`, `show_group_labels`, the axis titles, the `error` band and the `color` series.

The class and the helper `validate` use three private helpers of `Reporter`: `_resolve_query`, `_series_label` and `_path_name`. The base class offers no public way to resolve a query without drawing it, so a reader who copies this code depends on methods that may change. `_report` is the documented extension hook.

Two helpers sit on top of it. `validate` resolves each query directly, so that the exception a stale query raises is shown instead of being logged and swallowed by `Reporter.report`; it tabulates how many series each query selects and how many points the first series holds. `draw` runs the real `report` path and fails when a query draws no finite series, which is the check that a figure is not empty.

In [ ]:
class FigureReporter(Reporter):
    """Reporter backend that draws each resolved query as a matplotlib figure."""

    def __init__(self) -> None:
        self.figures: dict[str, plt.Figure] = {}
        self.series_counts: dict[str, int] = {}

    def _report(self, query, x, ys, errors, colors) -> None:
        figure, axes = plt.subplots(figsize=(6.0, 2.8))
        modes = query.plot_modes or ("lines+markers",) * len(query.y_paths)
        labels = query.legend_labels or (None,) * len(query.y_paths)
        styles = {"lines": "-", "markers": "o", "lines+markers": "o-"}
        drawn = 0

        for path, resolved, bands, mode, label in zip(
            query.y_paths, ys, errors, modes, labels
        ):
            named = False
            for group, values in resolved.items():
                xs = np.asarray(x[()] if () in x else x[group], dtype=float)
                y = np.asarray(values, dtype=float)
                if not np.isfinite(y).any():
                    continue
                name = self._series_label(path, group, label=label)
                if not query.show_group_labels and label is not None:
                    name, named = (label if not named else "_nolegend_"), True
                if colors is not None:
                    shade = colors[()] if () in colors else colors[group]
                    cmap = (query.colorscale or "viridis").lower()
                    points = axes.scatter(
                        xs, y, c=np.asarray(shade, dtype=float), cmap=cmap, s=16
                    )
                else:
                    line = axes.plot(xs, y, styles[mode], ms=3.5, label=name)[0]
                    if group in bands:
                        spread = np.asarray(bands[group], dtype=float)
                        axes.fill_between(
                            xs,
                            y - spread,
                            y + spread,
                            color=line.get_color(),
                            alpha=0.2,
                        )
                drawn += 1

        if colors is not None and drawn:
            figure.colorbar(points, ax=axes, label=query.color_label)
        axes.set_title(query.title, fontsize=9)
        axes.set_xlabel(query.x_label or self._path_name(query.x), fontsize=8)
        axes.set_ylabel(query.y_label or "value", fontsize=8)
        axes.tick_params(labelsize=7)
        if axes.get_legend_handles_labels()[0] and drawn <= 8:
            axes.legend(fontsize=7)
        figure.tight_layout()
        plt.close(figure)  # shown explicitly by `draw`, not by the notebook backend

        self.figures[query.title] = figure
        self.series_counts[query.title] = drawn

    def close(self) -> None:
        plt.close("all")


def validate(queries, metrics) -> pd.DataFrame:
    """Resolve each query against a history and tabulate what it selects."""
    reporter = FigureReporter()
    rows = []
    for query in queries:
        try:
            x, ys, errors, _ = reporter._resolve_query(metrics, query)
        except Exception as error:
            rows.append(
                {
                    "query": query.title,
                    "series": 0,
                    "points": 0,
                    "band": False,
                    "status": f"{type(error).__name__}: {error}",
                }
            )
            continue
        first = next(iter(ys[0].values()), [])
        rows.append(
            {
                "query": query.title,
                "series": sum(len(y) for y in ys),
                "points": len(first),
                "band": any(errors),
                "status": "ok",
            }
        )
    return pd.DataFrame(rows)


def draw(queries, metrics, *, show=None) -> FigureReporter:
    """Report the queries through the real ``report`` path and display figures.

    Parameters
    ----------
    queries : sequence of Query
        Queries to render.
    metrics : MetricSchema
        A peeked history to resolve them against.
    show : sequence of str, optional
        Titles of the figures to display; all of them when omitted.

    Returns
    -------
    FigureReporter
        The reporter, holding every figure.

    Raises
    ------
    AssertionError
        If a query drew no finite series.
    """
    reporter = FigureReporter()
    reporter.add_query(*queries)
    reporter.report(metrics)
    empty = [q.title for q in queries if reporter.series_counts.get(q.title, 0) == 0]
    assert not empty, f"queries that drew nothing: {empty}"
    for title in show if show is not None else reporter.figures:
        display(reporter.figures[title])
    return reporter

Back to the field added in Section 4: `validate` resolves the query on the new leaf against the history built there, and the check confirms that the leaf is addressable and holds the three pushed points.

In [ ]:
extended_table = validate([extended_query], extended_history)
display(extended_table)
assert extended_table.status.iloc[0] == "ok"
assert extended_table.series.iloc[0] == 1 and extended_table.points.iloc[0] == 3

## 7. Environment queries

These queries run on one `FisheryMetricSchema` history, so the x axis is the step of the episode and every path is a field of the schema. The first group describes the stock; the last three show the per-fisher fields, which live under the dynamic node `by_agent`. `SERIES` draws one curve per fisher, and `MEAN` with an `error_path` draws the mean over fishers with a standard-deviation band, so the queries do not need to know how many fishers there are or what their identifiers are.

The fields that exist but are not filled by this environment are not queried. In the fishery environment, the per-agent `action`, `observation`, `reward` and similar generic step fields stay empty; only `requested_harvest` and `delivered_harvest` are pushed per fisher, and the reward is available as the episode-level `reward_mean`.

In [ ]:
ENV_QUERIES = (
    Query(
        title="Fish stock and its reference point",
        x=("iter",),
        y=(("fish_stock",), ("B_msy",)),
        legend_labels=("stock B(t)", "biomass at MSY"),
        plot_modes=("lines+markers", "lines"),
        x_label="step",
        y_label="biomass",
    ),
    Query(
        title="Normalized fish biomass after each step",
        x=("iter",),
        y=("fish_norm_next_mean",),
        x_label="step",
        y_label="fraction of K",
    ),
    Query(
        title="Growth and process noise",
        x=("iter",),
        y=(("growth",), ("growth_noise",)),
        legend_labels=("growth plus noise", "noise only"),
        x_label="step",
        y_label="biomass per step",
    ),
    Query(
        title="Attempted and realized harvest",
        x=("iter",),
        y=(("H_attempted",), ("H_realized",), ("MSY",)),
        legend_labels=("attempted", "realized", "MSY"),
        plot_modes=("lines+markers", "lines+markers", "lines"),
        x_label="step",
        y_label="biomass per step",
    ),
    Query(
        title="Harvest as a share of K",
        x=("iter",),
        y=("total_usage_norm",),
        x_label="step",
    ),
    Query(
        title="Mean reward per step", x=("iter",), y=("reward_mean",), x_label="step"
    ),
    Query(
        title="Requested harvest per fisher",
        x=("iter",),
        y=("by_agent", SERIES, "requested_harvest"),
        legend_labels=("requested",),
        x_label="step",
        y_label="biomass per step",
    ),
    Query(
        title="Delivered harvest per fisher",
        x=("iter",),
        y=("by_agent", SERIES, "delivered_harvest"),
        legend_labels=("delivered",),
        x_label="step",
        y_label="biomass per step",
    ),
    Query(
        title="Mean delivered harvest over fishers",
        x=("iter",),
        y=("by_agent", MEAN, "delivered_harvest"),
        legend_labels=("mean delivered",),
        error="std",
        error_path=("by_agent",),
        x_label="step",
        y_label="biomass per step",
    ),
)

env_table = validate(ENV_QUERIES, env_history)
display(env_table)
assert (env_table.status == "ok").all()
# Series per query: the number of y paths, except that the per-fisher paths
# expand to one series per fisher.
assert env_table.series.tolist() == [2, 1, 2, 3, 1, 1, N_FISHERS, N_FISHERS, 1]

env_reporter = draw(
    ENV_QUERIES,
    env_history,
    show=(
        "Fish stock and its reference point",
        "Attempted and realized harvest",
        "Requested harvest per fisher",
        "Mean delivered harvest over fishers",
    ),
)

## 8. Inner optimizer queries

These queries run on the `RaySchema` of one inner training run, so the x axis is the training iteration, the root `iter` field. Three families are useful.

**Aggregates and performance** read plain fields: `("train", "rollout", "aggregate", "reward_mean")` and its siblings, and `("train", "performance", "<counter>")`. Several y paths in one query draw several curves in one plot, such as the minimum, mean and maximum return.

**Per-mechanism rollouts** read the episode fields below `by_mechanism`. Two reductions are chained: `SERIES` on `by_mechanism` keeps one curve per candidate, `MEAN` on `by_seed` averages the seeds of the environments (the training seeds in `train`, the evaluation seeds in `eval`), and `MEAN` on `by_episode` averages the episodes of each seed. The standard-deviation band is taken across seeds, so `error_path` stops at `by_seed`. A path ending on `fish_norm_next_mean`, `H_realized` or `reward_mean` is a scalar per episode and iteration. The mechanism stays the plotted group, while the seeds and the episodes are the dimensions that are reduced.

**Learner statistics** read `PolicyLearnerSchema` fields below `by_mechanism`, `by_seed` (keyed there by the training seed of the policy) and `by_policy`. The policy identifiers are runtime values, so `SERIES` on `by_policy` draws one curve per policy without naming it. These statistics are sparse in a real APPO run: RLlib reduces them only once every 20 gradient updates, and the adaptor fills the iterations in between with NaN so that the series keep one entry per iteration. A short real run therefore draws nothing for the learner queries. In a measured run of 23 iterations, the policy loss was NaN on iterations 0 to 19, had a value on iteration 20 and was NaN again afterwards. The made-up data below has finite values everywhere.

Two small helpers build the long paths.

In [ ]:
def per_mechanism(phase: str, field: str) -> MetricPath:
    """Path to an episode field averaged over seeds and episodes, per mechanism."""
    return (
        phase,
        "rollout",
        "by_mechanism",
        SERIES,
        "by_seed",
        MEAN,
        "by_episode",
        MEAN,
        field,
    )


def seed_dimension(phase: str) -> MetricPath:
    """Path of the dynamic node whose branches are averaged, for a std band."""
    return (phase, "rollout", "by_mechanism", SERIES, "by_seed")


def learner_path(field: str) -> MetricPath:
    """Path to a learner statistic averaged over seeds, per mechanism and policy."""
    return (
        "train",
        "learner",
        "by_mechanism",
        SERIES,
        "by_seed",
        MEAN,
        "by_policy",
        SERIES,
        field,
    )


LEARNER_DIMENSION = ("train", "learner", "by_mechanism", SERIES, "by_seed")

RAY_QUERIES = (
    Query(
        title="Train episode return",
        x=("iter",),
        y=tuple(
            ("train", "rollout", "aggregate", f"reward_{stat}")
            for stat in ("mean", "min", "max")
        ),
        legend_labels=("mean", "min", "max"),
        x_label="training iteration",
        y_label="episode return",
    ),
    Query(
        title="Train episode count",
        x=("iter",),
        y=(
            ("train", "rollout", "aggregate", "num_episodes"),
            ("train", "rollout", "aggregate", "num_episodes_lifetime"),
        ),
        legend_labels=("this iteration", "lifetime"),
        x_label="training iteration",
    ),
    Query(
        title="Environment steps per iteration",
        x=("iter",),
        y=(
            ("train", "performance", "env_steps_this_iter"),
            ("train", "performance", "agent_steps_this_iter_sum"),
        ),
        legend_labels=("environment steps", "agent steps"),
        x_label="training iteration",
    ),
    Query(
        title="Time per training iteration",
        x=("iter",),
        y=(
            ("train", "performance", "training_iteration_s"),
            ("train", "performance", "sample_s"),
            ("train", "performance", "learner_update_s"),
        ),
        legend_labels=("iteration", "sampling", "learner update"),
        x_label="training iteration",
        y_label="seconds",
    ),
    Query(
        title="Normalized fish biomass per mechanism",
        x=("iter",),
        y=per_mechanism("train", "fish_norm_next_mean"),
        legend_labels=("biomass",),
        error="std",
        error_path=seed_dimension("train"),
        x_label="training iteration",
        y_label="fraction of K",
    ),
    Query(
        title="Realized harvest per mechanism",
        x=("iter",),
        y=per_mechanism("train", "H_realized"),
        legend_labels=("harvest",),
        error="std",
        error_path=seed_dimension("train"),
        x_label="training iteration",
        y_label="biomass per step",
    ),
    Query(
        title="Episode return per mechanism",
        x=("iter",),
        y=per_mechanism("train", "reward_mean"),
        legend_labels=("return",),
        error="std",
        error_path=seed_dimension("train"),
        x_label="training iteration",
    ),
    *(
        Query(
            title=f"Learner {field} per mechanism",
            x=("iter",),
            y=learner_path(field),
            legend_labels=(field,),
            error="std",
            error_path=LEARNER_DIMENSION,
            x_label="training iteration",
        )
        for field in ("value_loss", "total_loss", "policy_entropy")
    ),
)

ray_table = validate(RAY_QUERIES, ray_history)
display(ray_table)
assert (ray_table.status == "ok").all()
# Three or two y paths for the plain queries, one series per candidate for the
# others (the made-up run has one policy, so the learner queries have one series
# per candidate too).
assert ray_table.series.tolist() == [3, 2, 2, 3] + [N_MECHANISMS] * 6

ray_reporter = draw(
    RAY_QUERIES,
    ray_history,
    show=(
        "Train episode return",
        "Normalized fish biomass per mechanism",
        "Learner value_loss per mechanism",
    ),
)

The table shows one series per candidate for the per-mechanism queries and a band on every query that declares one. The learner queries show one series per mechanism because the made-up run has a single policy; a run with several policies would add one series per policy.

### Train and eval at the inner level

An evaluation pass runs once at the end of `RayOptimizer.train`, so at the inner level every `eval` leaf holds a single point while `iter` holds one point per training iteration. A query that pairs an `eval` path with `x=("iter",)` has series of unequal length, and the reporter rejects it. The next cell shows the error. The fishery example therefore compares train and eval at the outer level, where each generation contributes one reduced point of both, which is the subject of the next section.

In [ ]:
eval_at_inner_level = Query(
    title="Eval return at the inner level",
    x=("iter",),
    y=("eval", "rollout", "aggregate", "reward_mean"),
)
mismatch = validate([eval_at_inner_level], ray_history)
print(mismatch.status.iloc[0])
assert mismatch.status.iloc[0].startswith("ValueError")

## 9. Outer optimizer queries

These queries run on the `ESSchema`, whose x axis is the generation. The schema stores the generation twice, in `iter` and in `generation`, and either can be the x path. Three kinds of data are available.

- **Candidate fitness.** `by_mechanism` is keyed by the index of the candidate in the population. A plain `SERIES` over it draws one marker series per index. The index is only a position in the generation's population: the search samples a fresh population every generation, so candidate 2 of one generation has no relation to candidate 2 of the next, and the cloud of markers is a picture of the fitness distribution per generation, not of four persistent individuals. The generation mean (`MEAN` with a band across candidates), the generation best and the best so far are plain series.
- **Fitness against a parameter.** One query can pair the parameter value and the fitness of every candidate. The x path and the y path both go through `by_mechanism` with `SERIES`, so the reporter pairs candidate by candidate: the groups of x and y must match, and the points of candidate `i` are its parameter value and its fitness. The `color` path colours the points by generation.
- **Search distribution.** `search_mean`, `global_best` and `generation_best` are keyed by parameter name, so `SERIES` on them draws one curve per optimized parameter, whatever the mechanism space is.

In [ ]:
ES_OWN_QUERIES = (
    Query(
        title="Candidate fitness and generation statistics",
        x=("generation",),
        y=(
            ("by_mechanism", SERIES, "fitness"),
            ("by_mechanism", MEAN, "fitness"),
            ("fitness_best",),
            ("best_fitness_global",),
        ),
        legend_labels=(
            "candidates",
            "generation mean",
            "generation best",
            "best so far",
        ),
        plot_modes=("markers", "lines+markers", "lines+markers", "lines"),
        show_group_labels=False,
        error="std",
        error_path=("by_mechanism",),
        x_label="generation",
        y_label="fitness",
    ),
    Query(
        title="Candidate fitness against the quota parameter",
        x=("by_mechanism", SERIES, "by_parameter", "quota", "value"),
        y=("by_mechanism", SERIES, "fitness"),
        plot_modes=("markers",),
        legend_labels=("candidates",),
        show_group_labels=False,
        color=("generation",),
        color_label="generation",
        colorscale="Viridis",
        x_label="normalized quota",
        y_label="fitness",
    ),
    Query(
        title="Search mean and best candidates per parameter",
        x=("generation",),
        y=(
            ("search_mean", SERIES, "value"),
            ("global_best", SERIES, "value"),
            ("generation_best", SERIES, "value"),
        ),
        legend_labels=("search mean", "best so far", "generation best"),
        show_group_labels=False,
        x_label="generation",
        y_label="normalized parameter",
    ),
    Query(
        title="Search width",
        x=("generation",),
        y=("sigma",),
        x_label="generation",
        y_label="sigma",
    ),
)

es_table = validate(ES_OWN_QUERIES, es_history)
display(es_table)
assert (es_table.status == "ok").all()
# Candidates, plus the mean, the best and the best so far; one series per
# candidate; one series per parameter and path; one series.
assert es_table.series.tolist() == [N_MECHANISMS + 3, N_MECHANISMS, 3, 1]

es_reporter = draw(ES_OWN_QUERIES, es_history)

The outer level can also read the inner optimizer's metrics, because `ESSchema.inner` is the `RaySchema` of the inner run reduced to one value per leaf and pushed once per generation. Because `RaySchema` reduces with `LAST`, that value is the one of the last inner iteration, not a mean over the iterations. Below `inner`, the same per-mechanism paths as before apply, with the generation as the x axis. Train and eval can now share an x axis, since both hold one point per generation. A query has a single `error_path`, so a band is drawn for one phase per query, and the two phases are two queries.

In [ ]:
ES_INNER_QUERIES = tuple(
    Query(
        title=f"{phase.capitalize()} normalized fish biomass per generation",
        x=("iter",),
        y=("inner", *per_mechanism(phase, "fish_norm_next_mean")),
        legend_labels=("biomass",),
        error="std",
        error_path=("inner", *seed_dimension(phase)),
        x_label="generation",
        y_label="fraction of K",
    )
    for phase in ("train", "eval")
)

es_inner_table = validate(ES_INNER_QUERIES, es_history)
display(es_inner_table)
assert (es_inner_table.status == "ok").all()
assert es_inner_table.series.tolist() == [N_MECHANISMS, N_MECHANISMS]

es_inner_reporter = draw(ES_INNER_QUERIES, es_history)

## 10. The query sets shipped with the fishery example

`examples/bilevel_fishery/queries.py` holds the queries that `debug.py` and the fishery tutorial attach: `ES_QUERIES` for the outer level, `INNER_QUERIES` for the inner level and `FISHERY_ENV_QUERIES` for the environment. They are written with the same tokens as above. The cell checks that each set resolves against a history with the real shapes, and draws one query of each.

In [ ]:
shipped = (
    ("ES_QUERIES", ES_QUERIES, es_history),
    ("INNER_QUERIES", INNER_QUERIES, ray_history),
    ("FISHERY_ENV_QUERIES", FISHERY_ENV_QUERIES, env_history),
)

for name, queries, history in shipped:
    table = validate(queries, history)
    print(f"{name}: {len(queries)} queries")
    display(table)
    assert (table.status == "ok").all()
    assert (table.series > 0).all()

_ = draw(ES_QUERIES, es_history, show=("Final train vs eval episode return",))
_ = draw(
    INNER_QUERIES,
    ray_history,
    show=("Mean normalized fish biomass over training episodes",),
)

## 11. What a stale path looks like

A stale path fails in the resolution step, and the reporter then logs the error and skips only that query, so the rest of the run is not affected. The cell below resolves one query of each common kind of mistake with `validate`, which keeps the message instead of logging it. They are the failures to expect when a query is copied from an older description of the schema.

- A field that is not in the schema, such as `fish_norm`, raises an unknown-path error. The normalized biomass of the present schema is `fish_norm_next_mean` and its siblings.
- `by_seed` has no `aggregate`. The mean over the episodes of a seed is a `MEAN` token on `by_episode`, as in the per-mechanism queries.
- A dynamic node needs a token or a key. Forgetting the token before the field name makes the field look like a key.
- A path that stops before a leaf, and a path that ends on a `*_series` leaf of the inner optimizer (a list of lists), are both rejected.
- A reduction token cannot be applied to a schema node, only to a dictionary node.

In [ ]:
stale = {
    "field not in the schema": (
        Query(title="stale 1", x=("iter",), y=("fish_norm",)),
        env_history,
        "KeyError",
    ),
    "aggregate below a seed": (
        Query(
            title="stale 2",
            x=("iter",),
            y=(
                "train",
                "rollout",
                "by_mechanism",
                SERIES,
                "by_seed",
                MEAN,
                "aggregate",
                "fish_norm_next_mean",
            ),
        ),
        ray_history,
        "KeyError",
    ),
    "token missing on a dynamic node": (
        Query(title="stale 3", x=("iter",), y=("by_agent", "requested_harvest")),
        env_history,
        "KeyError",
    ),
    "path stops before a leaf": (
        Query(title="stale 4", x=("iter",), y=("train", "rollout", "aggregate")),
        ray_history,
        "KeyError",
    ),
    "nested series": (
        Query(
            title="stale 5",
            x=("iter",),
            y=per_mechanism("train", "fish_norm_next_series"),
        ),
        ray_history,
        "ValueError",
    ),
    "token on a schema node": (
        Query(
            title="stale 6",
            x=("iter",),
            y=("train", SERIES, "rollout", "aggregate", "reward_mean"),
        ),
        ray_history,
        "TypeError",
    ),
}

for reason, (query, history, expected) in stale.items():
    status = validate([query], history).status.iloc[0]
    print(f"{reason}:\n    {status[:150]}")
    assert status.startswith(f"{expected}:"), status

## 12. The CSV backend and the reporting lifecycle

### CSV

`CSVReporter` writes one long-form file per query, named after the query title, and rewrites it at every report, so the file always holds the full history of the latest schema it received. The columns are `query`, `x`, `series`, `value`, `error` and `color`; the error and colour cells are empty when the query has none. A query that resolves to no series writes no file and leaves any earlier file untouched. Two queries whose titles reduce to the same file name share a file. The cell below reports the shipped outer queries through a CSV reporter into a temporary directory and loads one file.

The TensorBoard reporter writes the same queries as event files under `<log_dir>/<project>/<world>-<label>`, and needs the optional `tensorboard` package (`uv sync --extra tensorboard`). The Weights & Biases reporter starts one run per owner and draws the plots, including the colour series and the plot modes, which the other two backends do not draw. All three consume the same query objects, so a query does not change when the backend does.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    config = CSVConfig(project="viz_tutorial", output_dir=directory)
    config.world = "synthetic"
    csv_reporter = config.build(label="ES")
    csv_reporter.add_query(*ES_QUERIES)
    csv_reporter.report(es_history)

    files = sorted(csv_reporter.output_dir.glob("*.csv"))
    print(f"{len(files)} files for {len(ES_QUERIES)} queries:")
    for file in files:
        print("   ", file.name, "-", len(pd.read_csv(file)), "rows")

    table = pd.read_csv(csv_reporter.path_for(ES_QUERIES[0]))
    display(table.head(8))

assert len(files) == len(ES_QUERIES)

### Lifecycle

The three levels use the two read operations at different moments.

- **Environment.** At the end of every episode, the callback `log_and_report_episode_metrics` peeks the environment's logger, passes the history to the environment's own reporter (the environment-level queries), reduces the logger, and hands the reduced episode upward tagged with an episode prefix of the form `env=<index>|m=<mechanism>|ps=<policy seed>|ss=<seed>`. The Ray adaptor files the episode under `by_episode` as `<prefix>|n=<position>`, where the position counts the episodes that sub-environment ended in the iteration.
- **Inner optimizer.** `RayOptimizer.train` pushes the iteration index and one `train` payload per iteration, runs the evaluation, which pushes one `eval` payload, and then reports once by peeking its logger. The history it reports therefore has one point per training iteration for `train` and a single point for `eval`.
- **Outer optimizer.** After every generation, `ESOptimizer` pushes one `ESSchema` payload, whose `inner` branch is the reduced metrics of that generation's inner run, and reports by peeking. The ES logger is never reduced during the search, which is why the plots are cumulative.

The logger, not the reporter, owns the history, so a backend does not need its own cache of earlier points.

## 13. Limits of the present design

- **One error band per query.** A `Query` has a single `error_path`, and the band attaches to the y paths that pass through the `MEAN` token it names. Train and eval bands therefore need two queries, as in the outer-level example.
- **Inner-level eval series have one point.** They cannot share the `iter` axis of the training series, as shown above.
- **Plot styling is per query, not per figure.** `plot_modes` and `legend_labels` style the y paths of a query; the colour series, the axis titles and the plot modes are drawn by the W&B backend only, and the CSV backend writes the colour as a column.
- **No multidimensional plots.** A parallel-coordinates plot of every evaluated candidate (one line per candidate, one axis per parameter, fitness as the last axis and as the colour) cannot be expressed by a query, which selects series against one x path. The data it needs is in `ESSchema.by_mechanism`, but no query class for it exists in `core/reporting`.
- **Candidate indices are positions.** `by_mechanism` keys in the outer schema are indices in the generation's population, not persistent identities.

## 14. Design rules for new metrics

Before adding a visualization, answer four questions.

1. **Which entity owns the metric?** The environment episode, an agent, a policy and its learner, the optimizer's performance, or an ES generation, candidate or parameter. The owner decides the schema class and therefore the path.
2. **What should the reduction be?** `SERIES` if the history itself is the metric, and `MEAN`, `MIN`, `MAX`, `SUM` or `LAST` for a compiled episode or iteration summary.
3. **Does the key exist at runtime?** A static field is declared on the schema. A runtime identifier (mechanism, seed, episode, policy, agent, parameter) goes in a `dict[ID, MetricSchema]` branch, and queries reach it with `SERIES` or `MEAN`.
4. **Is the picture one-dimensional?** A line or scatter, with one series per key when needed, is a `Query`. A table-like or multidimensional plot is not expressible today.

Then check the new query on a populated history with `validate` and `draw`, as in this notebook, before attaching it to an optimizer. This keeps collection, aggregation, selection and presentation separate.

## References

- Pella, J. J., & Tomlinson, P. K. (1969). A generalized stock production model. *Bulletin of the Inter-American Tropical Tuna Commission*, 13(3), 421-458.
- Schaefer, M. B. (1954). Some aspects of the dynamics of populations important to the management of the commercial marine fisheries. *Bulletin of the Inter-American Tropical Tuna Commission*, 1(2), 27-56.